In [2]:
import torch
from torchvision import datasets, transforms

# Convert images to PyTorch tensors and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Training dataset
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Testing dataset
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

100%|██████████████████████████████████████| 9.91M/9.91M [00:15<00:00, 629kB/s]
100%|█████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 89.1kB/s]
100%|██████████████████████████████████████| 1.65M/1.65M [00:04<00:00, 352kB/s]
100%|█████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 3.36MB/s]

Train: 60000
Test: 10000


In [4]:
from torch.utils.data import  DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)



In [5]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [7]:
import torch.nn as nn

class Handwritten(nn.Module):
    def __init__(self):
        super(Handwritten, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size = 3, padding = 1),#channel, output feature maps, filter_size, padding
            nn.ReLU(),
            nn.MaxPool2d(2, 2),#window size, sride

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),#input channel, output feature maps, filter_size, padding
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),#channel, output feature maps, filter_size, padding
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )      
        
        self.fc_layers = nn.Sequential(
            nn.Linear(128*3*3, 256),
            nn.ReLU(),

            nn.Linear(256,10)
            
        )
    def forward(self, x):
        x = self.conv_layers(x)
        x = torch.flatten(x, 1)
        x = self.fc_layers(x)
        return x
        

In [8]:
model = Handwritten()

In [10]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [11]:
for epoch in range(10):
    model.train()
    running_loss = 0
    for img, label in train_loader:
        optimizer.zero_grad()
        output = model(img)
        loss = criterion(output,label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(running_loss/len(train_loader))

0.1368731905284004
0.04018087156833326
0.026174767376499566
0.02328916841582277
0.01737180608026842
0.014422538821722975
0.01179905539304337
0.01039322108145153
0.010938143077175175
0.008743667735255284


In [12]:
correct = 0
total = 0

with torch.no_grad():
    for img, label in test_loader:
        output = model(img)
        _,pred  = torch.max(output,1)
        correct += (pred == label).sum().item()
        total += label.size(0)
    print(correct/total)

0.9913
